# FockPARFLM v2 — Dyck₂ Falsifier with Q/K/V-Structured Creation

## Motivation

This notebook implements the **F2 experiment** from
[`docs/Improving_the_Fock_Mechanism_to_match_Attention.md`](../../../../docs/Improving_the_Fock_Mechanism_to_match_Attention.md) §12.1.

The FockPARFLM v1 (mean-conditioned creation gate) achieved only +1.3 pp over
the PARFLM baseline on the Dyck₂ deep-test (39.2% vs 37.9%).  The structural
diagnosis (§§7–8) identifies the root cause: the v1 gate lacks the three
properties of attention (asymmetry, Q/K/V decoupling, competitive normalisation).

FockPARFLM v2 replaces the mean-conditioned gate with a Q/K/V-structured creation
protocol and adds a non-conservative reverse channel (§10).  This notebook runs
a controlled three-arm comparison:

| Arm | Architecture | Expected result |
|-----|-------------|----------------|
| `F2-baseline` | PARFLM (no registers) | ~37.9% deep-test acc (replication) |
| `F2-fock-v1` | FockPARFLM v1 (mean gate) | ~39.2% (replication) |
| **`F2-fock-v2`** | **FockPARFLM v2 (Q/K/V gate + reverse channel)** | **>50% deep-test acc (prediction)** |

**Success criterion:** F2-fock-v2 achieves >50% deep-test accuracy at depth 5–12,
with 3/3 seed consistency.

## Cell structure

Run the notebook once per `CELL` value.  All three arms use the same data split,
hyperparameters, and evaluation protocol.  Only the model architecture changes.

## 0. Environment setup + cell selector

In [ ]:
CELL = 'F2-fock-v2'   # one of: 'F2-baseline' | 'F2-fock-v1' | 'F2-fock-v2'
SEED = 0

REPO_URL        = 'https://github.com/dimitarpg13/semsimula.git'
REPO_BRANCH     = 'main'
COLAB_REPO_PATH = '/content/semsimula'
GDRIVE_OUT_REL  = 'semsimula_fock_v2'

import os, sys, shutil, subprocess, json, time
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print(f'IN_COLAB = {IN_COLAB}')


def _sh(cmd: str) -> None:
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise RuntimeError(f'command failed (exit {r.returncode}): {cmd}')


if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    GDRIVE_OUT = Path('/content/drive/MyDrive') / GDRIVE_OUT_REL
    GDRIVE_OUT.mkdir(parents=True, exist_ok=True)
    print(f'GDrive output root = {GDRIVE_OUT}')

    REPO_ROOT = Path(COLAB_REPO_PATH)
    if not (REPO_ROOT / '.git').exists():
        if REPO_ROOT.exists():
            shutil.rmtree(REPO_ROOT)
        _sh(f'git clone --depth=1 -b {REPO_BRANCH} {REPO_URL} {COLAB_REPO_PATH}')
    else:
        _sh(f'cd {COLAB_REPO_PATH} && git pull --ff-only')
else:
    REPO_ROOT = Path('__file__').resolve().parents[3]
    if not (REPO_ROOT / 'notebooks').exists():
        REPO_ROOT = Path.cwd()
        while not (REPO_ROOT / 'notebooks').exists() and REPO_ROOT != REPO_ROOT.parent:
            REPO_ROOT = REPO_ROOT.parent
    GDRIVE_OUT = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'parf' / 'results' / 'fock_v2'
    GDRIVE_OUT.mkdir(parents=True, exist_ok=True)

PARF_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'parf'
sys.path.insert(0, str(PARF_DIR))
CONSERV_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch'
sys.path.insert(0, str(CONSERV_DIR))

print(f'REPO_ROOT = {REPO_ROOT}')
print(f'PARF_DIR  = {PARF_DIR}')
print(f'CELL      = {CELL}')
print(f'SEED      = {SEED}')
print(f'Output    = {GDRIVE_OUT}')

## 1. Imports

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW

from dyck_data import (
    DyckConfig,
    generate_dyck_dataset,
    generate_depth_controlled_dataset,
    get_dyck_batch,
)
from model_parf import PARFConfig, PARFLM
from model_parf_sparse import SparsePARFConfig, SparsePARFLM
from model_fock_parf import FockPARFConfig, FockPARFLM
from model_fock_parf_v2 import FockPARFConfig_v2, FockPARFLM_v2

DEVICE = (
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

## 2. Experimental configuration

In [ ]:
# ── Dyck₂ data configuration ──────────────────────────────────────
DYCK_CFG = DyckConfig(
    n_types=2,
    max_depth=12,
    min_length=8,
    max_length=64,
    p_open=0.55,
)

N_TRAIN = 20_000
N_VAL   = 2_000
DEEP_TEST_DEPTH_MIN = 5
DEEP_TEST_DEPTH_MAX = 12
N_DEEP_TEST = 2_000

# ── Training hyperparameters ──────────────────────────────────────
TOTAL_STEPS   = 4_000
BATCH_SIZE    = 64
LR            = 3e-4
WEIGHT_DECAY  = 0.01
EVAL_EVERY    = 200
LOG_EVERY     = 50

# ── Shared model hyperparameters (match F1 experiment) ────────────
D       = 64
L       = 4
V_HIDDEN = 64
V_DEPTH  = 2
TOP_K    = 8

# ── Fock-specific ─────────────────────────────────────────────────
M_REGISTERS  = 16
D_K          = 32    # Key/query dim for v2 creation gate
DECAY        = 0.9
THRESHOLD    = 0.1

print(f'Dyck_{DYCK_CFG.n_types} | max_depth={DYCK_CFG.max_depth} | '
      f'vocab_size={DYCK_CFG.vocab_size}')
print(f'd={D}, L={L}, M={M_REGISTERS}, d_k={D_K}')
print(f'{TOTAL_STEPS} steps, batch={BATCH_SIZE}, lr={LR}')

## 3. Data generation

In [ ]:
torch.manual_seed(SEED)
np.random.seed(SEED)

x_train, y_train = generate_dyck_dataset(DYCK_CFG, N_TRAIN, seed=SEED)
x_val,   y_val   = generate_dyck_dataset(DYCK_CFG, N_VAL,   seed=SEED + 1000)

x_deep, y_deep, depths_deep = generate_depth_controlled_dataset(
    DYCK_CFG, N_DEEP_TEST,
    min_depth=DEEP_TEST_DEPTH_MIN,
    max_depth=DEEP_TEST_DEPTH_MAX,
    seed=SEED + 2000,
)

print(f'Train: {x_train.shape}, Val: {x_val.shape}')
print(f'Deep test: {x_deep.shape}  depth range [{DEEP_TEST_DEPTH_MIN}, {DEEP_TEST_DEPTH_MAX}]')
print(f'Deep test depth distribution:')
for d in sorted(set(depths_deep)):
    print(f'  depth {d}: {(depths_deep == d).sum()} samples')

## 4. Model construction

In [ ]:
torch.manual_seed(SEED)

if CELL == 'F2-baseline':
    cfg = SparsePARFConfig(
        vocab_size=DYCK_CFG.vocab_size,
        d=D, max_len=DYCK_CFG.max_length + 2, L=L,
        v_hidden=V_HIDDEN, v_depth=V_DEPTH,
        v_phi_d_type=4, v_phi_d_angle=2,
        v_phi_phi_hidden=8, v_phi_theta_hidden=8,
        v_phi_mlp_hidden=16,
        mass_mode='global',
        top_k=TOP_K,
        score_head_hidden=8,
    )
    model = SparsePARFLM(cfg).to(DEVICE)
    arm_label = 'PARFLM baseline (no registers)'

elif CELL == 'F2-fock-v1':
    cfg = FockPARFConfig(
        vocab_size=DYCK_CFG.vocab_size,
        d=D, max_len=DYCK_CFG.max_length + 2, L=L,
        v_hidden=V_HIDDEN, v_depth=V_DEPTH,
        v_phi_d_type=4, v_phi_d_angle=2,
        v_phi_phi_hidden=8, v_phi_theta_hidden=8,
        v_phi_mlp_hidden=16,
        mass_mode='global',
        top_k=TOP_K,
        score_head_hidden=8,
        n_registers=M_REGISTERS,
        creation_gate_hidden=32,
        stack_discipline=True,
        register_salience_decay=DECAY,
        register_salience_threshold=THRESHOLD,
    )
    model = FockPARFLM(cfg).to(DEVICE)
    arm_label = f'FockPARFLM v1 (mean gate, M={M_REGISTERS}, stack)'

elif CELL == 'F2-fock-v2':
    cfg = FockPARFConfig_v2(
        vocab_size=DYCK_CFG.vocab_size,
        d=D, max_len=DYCK_CFG.max_length + 2, L=L,
        v_hidden=V_HIDDEN, v_depth=V_DEPTH,
        v_phi_d_type=4, v_phi_d_angle=2,
        v_phi_phi_hidden=8, v_phi_theta_hidden=8,
        v_phi_mlp_hidden=16,
        mass_mode='global',
        top_k=TOP_K,
        score_head_hidden=8,
        n_registers=M_REGISTERS,
        d_k=D_K,
        stack_discipline=True,
        register_salience_decay=DECAY,
        register_salience_threshold=THRESHOLD,
        destruction_gate_hidden=32,
        reverse_channel=True,
    )
    model = FockPARFLM_v2(cfg).to(DEVICE)
    arm_label = f'FockPARFLM v2 (Q/K/V gate + reverse channel, M={M_REGISTERS})'

else:
    raise ValueError(f'Unknown CELL={CELL!r}')

n_params = sum(p.numel() for p in model.parameters())
print(f'\n{arm_label}')
print(f'Total params: {n_params:,}')

if CELL == 'F2-fock-v2':
    overhead = model.get_fock_v2_overhead()
    print(f'Fock v2 overhead: {overhead:,} ({100*overhead/n_params:.1f}%)')
elif CELL == 'F2-fock-v1':
    overhead = model.get_register_overhead()
    print(f'Fock v1 overhead: {overhead:,} ({100*overhead/n_params:.1f}%)')

## 5. Evaluation helpers

In [ ]:
@torch.no_grad()
def evaluate(model, x_np, y_np, device, batch_size=256):
    """Compute mean loss and PPL on a dataset."""
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    n = len(x_np)
    for start in range(0, n, batch_size):
        xb = torch.from_numpy(x_np[start:start+batch_size]).to(device)
        yb = torch.from_numpy(y_np[start:start+batch_size]).to(device)
        with torch.enable_grad():
            _, loss = model(xb, targets=yb)
        mask = yb != -100
        n_tok = mask.sum().item()
        total_loss += loss.item() * n_tok
        total_tokens += n_tok
    avg_loss = total_loss / max(total_tokens, 1)
    return avg_loss, np.exp(avg_loss)


@torch.no_grad()
def deep_test_accuracy(model, x_np, y_np, device, batch_size=256):
    """Per-token accuracy on the deep test set.

    Measures whether the model can predict the correct next bracket
    at depth 5–12 nesting.  This is the primary expressivity diagnostic.
    """
    model.eval()
    correct = 0
    total = 0
    n = len(x_np)
    for start in range(0, n, batch_size):
        xb = torch.from_numpy(x_np[start:start+batch_size]).to(device)
        yb = torch.from_numpy(y_np[start:start+batch_size]).to(device)
        with torch.enable_grad():
            logits, _ = model(xb)
        preds = logits.argmax(dim=-1)
        mask = yb != -100
        correct += ((preds == yb) & mask).sum().item()
        total += mask.sum().item()
    return correct / max(total, 1)


print('Evaluation helpers defined.')

## 6. Training loop

In [ ]:
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
rng = np.random.default_rng(SEED)

train_log = []
best_deep_acc = 0.0
best_step = 0

OUT_DIR = GDRIVE_OUT / f'{CELL}_seed{SEED}'
OUT_DIR.mkdir(parents=True, exist_ok=True)
log_path = OUT_DIR / 'training_log.jsonl'

print(f'Training {CELL} for {TOTAL_STEPS} steps...')
print(f'Logging to: {log_path}')
print(f'{"":>6s}  {"train_loss":>10s}  {"val_loss":>8s}  {"val_ppl":>8s}  '
      f'{"deep_acc":>8s}  {"time_s":>6s}')
print('-' * 60)

t0 = time.time()
running_loss = 0.0
n_loss = 0

for step in range(1, TOTAL_STEPS + 1):
    model.train()

    xb_np, yb_np = get_dyck_batch(x_train, y_train, BATCH_SIZE, rng)
    xb = torch.from_numpy(xb_np).to(DEVICE)
    yb = torch.from_numpy(yb_np).to(DEVICE)

    _, loss = model(xb, targets=yb)

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    running_loss += loss.item()
    n_loss += 1

    if step % EVAL_EVERY == 0 or step == TOTAL_STEPS:
        avg_train = running_loss / n_loss
        running_loss = 0.0
        n_loss = 0

        val_loss, val_ppl = evaluate(model, x_val, y_val, DEVICE)
        deep_acc = deep_test_accuracy(model, x_deep, y_deep, DEVICE)
        elapsed = time.time() - t0

        record = {
            'step': step,
            'train_loss': avg_train,
            'val_loss': val_loss,
            'val_ppl': val_ppl,
            'deep_test_accuracy': deep_acc,
            'elapsed_s': round(elapsed, 1),
        }
        train_log.append(record)

        with open(log_path, 'a') as f:
            f.write(json.dumps(record) + '\n')

        if deep_acc > best_deep_acc:
            best_deep_acc = deep_acc
            best_step = step
            ckpt_path = OUT_DIR / 'best_model.pt'
            torch.save({
                'step': step,
                'model_state_dict': model.state_dict(),
                'deep_test_accuracy': deep_acc,
                'val_ppl': val_ppl,
                'config': cfg.__dict__,
            }, ckpt_path)

        print(f'{step:6d}  {avg_train:10.4f}  {val_loss:8.4f}  '
              f'{val_ppl:8.2f}  {deep_acc:8.4f}  {elapsed:6.0f}')

print(f'\nDone.  Best deep-test accuracy: {best_deep_acc:.4f} at step {best_step}')
print(f'Training log: {log_path}')

## 7. Training curves

In [ ]:
import matplotlib.pyplot as plt

steps = [r['step'] for r in train_log]
val_ppls = [r['val_ppl'] for r in train_log]
deep_accs = [r['deep_test_accuracy'] for r in train_log]
train_losses = [r['train_loss'] for r in train_log]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(steps, train_losses, 'b-', label='Train loss')
axes[0].plot(steps, [r['val_loss'] for r in train_log], 'r-', label='Val loss')
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Loss')
axes[0].set_title(f'{CELL} — Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(steps, val_ppls, 'g-')
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Val PPL')
axes[1].set_title(f'{CELL} — Perplexity')
axes[1].grid(True, alpha=0.3)

axes[2].plot(steps, deep_accs, 'm-')
axes[2].axhline(y=0.5, color='k', linestyle='--', alpha=0.5, label='50% target')
axes[2].set_xlabel('Step')
axes[2].set_ylabel('Deep-test Accuracy')
axes[2].set_title(f'{CELL} — Deep Test (depth {DEEP_TEST_DEPTH_MIN}–{DEEP_TEST_DEPTH_MAX})')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(OUT_DIR / f'{CELL}_seed{SEED}_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Final val PPL: {val_ppls[-1]:.2f}')
print(f'Final deep-test accuracy: {deep_accs[-1]:.4f}')
print(f'Best deep-test accuracy: {best_deep_acc:.4f} (step {best_step})')

## 8. Register diagnostics (v2 only)

When running the `F2-fock-v2` arm, inspect the register lifecycle:
- How many registers are active per layer?
- What are the attention weight distributions in the creation gate?
- How does salience evolve across layers?

In [ ]:
if CELL == 'F2-fock-v2':
    model.eval()

    xb_np, _ = get_dyck_batch(x_deep, y_deep, 8, np.random.default_rng(42))
    xb = torch.from_numpy(xb_np).to(DEVICE)

    B, T = xb.shape
    M = cfg.n_registers
    d = cfg.d

    with torch.enable_grad():
        h0 = model._embed(xb)
        r, salience = model._init_registers(B, h0.device)

        layer_saliences = []
        layer_n_active = []

        h = h0
        h_prev = h0
        m_b = model.compute_mass(xb)
        gamma, dt = model.gamma, cfg.dt

        for ell in range(cfg.L):
            h_new, h_prev_out, r, salience = model._fock_v2_layer_step(
                h, h_prev, r, salience, m_b, gamma, dt, layer_idx=ell,
            )
            h_prev = h_prev_out
            h = h_new

            active = model._active_mask(salience)
            layer_saliences.append(salience.detach().cpu().numpy())
            layer_n_active.append(active.sum(dim=-1).float().mean().item())

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].bar(range(cfg.L), layer_n_active)
    axes[0].set_xlabel('Layer')
    axes[0].set_ylabel('Mean active registers')
    axes[0].set_title('Active register count per layer')
    axes[0].set_xticks(range(cfg.L))

    sal_matrix = np.stack([s.mean(axis=0) for s in layer_saliences])  # (L, M)
    im = axes[1].imshow(sal_matrix, aspect='auto', cmap='viridis')
    axes[1].set_xlabel('Register index')
    axes[1].set_ylabel('Layer')
    axes[1].set_title('Mean salience (batch-averaged)')
    plt.colorbar(im, ax=axes[1])

    plt.tight_layout()
    fig.savefig(OUT_DIR / f'{CELL}_seed{SEED}_register_diag.png',
                dpi=150, bbox_inches='tight')
    plt.show()
else:
    print(f'Skipping register diagnostics for {CELL} (not a v2 arm).')

## 9. Cross-arm comparison dashboard

After running all three arms (`F2-baseline`, `F2-fock-v1`, `F2-fock-v2`),
load their training logs and produce a comparison plot.

In [ ]:
arms = ['F2-baseline', 'F2-fock-v1', 'F2-fock-v2']
arm_colors = {'F2-baseline': 'blue', 'F2-fock-v1': 'orange', 'F2-fock-v2': 'green'}
arm_labels = {
    'F2-baseline': 'PARFLM baseline',
    'F2-fock-v1':  'FockPARFLM v1 (mean gate)',
    'F2-fock-v2':  'FockPARFLM v2 (Q/K/V + reverse)',
}

# F1 reference values from the Phase 1 experiment
F1_REF = {
    'baseline': {'ppl': 3.50, 'deep_acc': 0.3793},
    'fock-v1':  {'ppl': 3.43, 'deep_acc': 0.3922},
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

found_any = False
for arm in arms:
    log_file = GDRIVE_OUT / f'{arm}_seed{SEED}' / 'training_log.jsonl'
    if not log_file.exists():
        print(f'  [{arm}] not found at {log_file} — skipping')
        continue
    found_any = True
    records = [json.loads(line) for line in open(log_file)]
    steps = [r['step'] for r in records]
    ppls = [r['val_ppl'] for r in records]
    accs = [r['deep_test_accuracy'] for r in records]

    axes[0].plot(steps, ppls, color=arm_colors[arm], label=arm_labels[arm])
    axes[1].plot(steps, accs, color=arm_colors[arm], label=arm_labels[arm])

    best_acc = max(accs)
    final_ppl = ppls[-1]
    print(f'  [{arm}] final PPL={final_ppl:.2f}, best deep-acc={best_acc:.4f}')

if found_any:
    axes[0].set_xlabel('Step')
    axes[0].set_ylabel('Val PPL')
    axes[0].set_title('Dyck₂ Validation Perplexity')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].axhline(y=0.5, color='k', linestyle='--', alpha=0.5, label='50% target')
    axes[1].set_xlabel('Step')
    axes[1].set_ylabel('Deep-test Accuracy')
    axes[1].set_title(f'Dyck₂ Deep Test (depth {DEEP_TEST_DEPTH_MIN}–{DEEP_TEST_DEPTH_MAX})')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    fig.savefig(GDRIVE_OUT / f'F2_cross_arm_comparison_seed{SEED}.png',
                dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No completed arms found yet.  Run each cell value first.')

## 10. Summary

| Arm | Architecture | Expected | Actual |
|-----|-------------|----------|--------|
| F2-baseline | PARFLM | ~37.9% | (fill after run) |
| F2-fock-v1 | FockPARFLM v1 (mean gate) | ~39.2% | (fill after run) |
| **F2-fock-v2** | **FockPARFLM v2 (Q/K/V)** | **>50%** | (fill after run) |

**Success criterion:** F2-fock-v2 achieves >50% deep-test accuracy
at depth 5–12, with 3/3 seed consistency.

**Next steps:**
- If F2-fock-v2 succeeds: scale up to `d=128, M=32, 8k steps` (F2-scale-up)
- If F2-fock-v2 fails: diagnose via the register diagnostics in cell 8;
  consider increasing `d_k`, `M`, or training duration.